In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

route_path = "../data/raw/ROUTE.csv"

route_df = pd.read_csv(route_path)

print("Route dataset loaded successfully.")
print("Shape:", route_df.shape)

print("\nColumns:")
print(route_df.columns.tolist())

print("\nFirst 5 records:")
print(route_df.head().to_string(index=False))

Route dataset loaded successfully.
Shape: (92, 9)

Columns:
['Route_ID', 'Depot_ID', 'Depot_Name', 'District', 'Origin_or_Depot', 'Destination_or_Corridor', 'Distance_KM', 'Terrain_Class', 'Terrain_Score']

First 5 records:
            Route_ID  Depot_ID Depot_Name           District Origin_or_Depot    Destination_or_Corridor  Distance_KM Terrain_Class  Terrain_Score
DEPOT-CORR-KSRTC-001 KSRTC-001      ADOOR     Pathanamthitta           ADOOR Depot operational corridor          NaN       Rolling           0.75
DEPOT-CORR-KSRTC-002 KSRTC-002  ALAPPUZHA          Alappuzha       ALAPPUZHA Depot operational corridor          NaN          Flat           1.00
DEPOT-CORR-KSRTC-003 KSRTC-003      ALUVA          Ernakulam           ALUVA Depot operational corridor          NaN  Flat/Rolling           0.90
DEPOT-CORR-KSRTC-004 KSRTC-004   ANKAMALY          Ernakulam        ANKAMALY Depot operational corridor          NaN  Flat/Rolling           0.90
DEPOT-CORR-KSRTC-005 KSRTC-005   ATTINGAL Thir

In [2]:
# Validate route dataset completeness

print("Missing values:")
print(route_df.isnull().sum())

print("\nTerrain distribution:")
print(route_df["Terrain_Class"].value_counts())

print("\nTerrain score distribution:")
print(route_df["Terrain_Score"].value_counts().sort_index())

print("\nNumber of unique depots:", route_df["Depot_ID"].nunique())
print("Number of unique districts:", route_df["District"].nunique())

Missing values:
Route_ID                    0
Depot_ID                    0
Depot_Name                  0
District                    0
Origin_or_Depot             0
Destination_or_Corridor     0
Distance_KM                92
Terrain_Class               0
Terrain_Score               0
dtype: int64

Terrain distribution:
Terrain_Class
Flat/Rolling    57
Rolling         18
Flat             7
Steep            7
Hilly            3
Name: count, dtype: int64

Terrain score distribution:
Terrain_Score
0.20     7
0.40     3
0.75    18
0.90    57
1.00     7
Name: count, dtype: int64

Number of unique depots: 92
Number of unique districts: 14


In [4]:
# Create terrain-based EV/Diesel recommendation

def terrain_recommendation(terrain):
    if terrain in ["Flat", "Flat/Rolling"]:
        return "EV Favorable"
    elif terrain == "Rolling":
        return "EV Conditional"
    elif terrain in ["Hilly", "Steep"]:
        return "Diesel Preferred"
    else:
        return "Review Required"


route_df["Terrain_Recommendation"] = (
    route_df["Terrain_Class"].apply(terrain_recommendation)
)

print("Terrain-based route screening:")
print(
    route_df[
        [
            "Depot_ID",
            "Depot_Name",
            "District",
            "Terrain_Class",
            "Terrain_Score",
            "Terrain_Recommendation"
        ]
    ].head(50).to_string(index=False)
)

print("\nRecommendation distribution:")
print(route_df["Terrain_Recommendation"].value_counts())

Terrain-based route screening:
 Depot_ID      Depot_Name           District Terrain_Class  Terrain_Score Terrain_Recommendation
KSRTC-001           ADOOR     Pathanamthitta       Rolling           0.75         EV Conditional
KSRTC-002       ALAPPUZHA          Alappuzha          Flat           1.00           EV Favorable
KSRTC-003           ALUVA          Ernakulam  Flat/Rolling           0.90           EV Favorable
KSRTC-004        ANKAMALY          Ernakulam  Flat/Rolling           0.90           EV Favorable
KSRTC-005        ATTINGAL Thiruvananthapuram  Flat/Rolling           0.90           EV Favorable
KSRTC-006 CHADAYAMANGALAM             Kollam  Flat/Rolling           0.90           EV Favorable
KSRTC-007      CHALAKKUDY           Thrissur  Flat/Rolling           0.90           EV Favorable
KSRTC-008   CHANGANASSERY           Kottayam       Rolling           0.75         EV Conditional
KSRTC-009      CHATHANOOR             Kollam  Flat/Rolling           0.90           EV Favorable

In [6]:
# Load depot-level ML predictions from FINALREVIEW

depot_predictions = pd.read_csv(
    "../outputs/FUTURE_DEPOT_PREDICTIONS.csv"
)

print("Depot predictions loaded successfully.")
print("Shape:", depot_predictions.shape)

print("\nColumns:")
print(depot_predictions.columns.tolist())

print("\nFirst 5 records:")
print(
    depot_predictions[
        [
            "Depot ID",
            "Depot Name",
            "District",
            "Predicted EV Suitability Score"
        ]
    ].head().to_string(index=False)
)

Depot predictions loaded successfully.
Shape: (92, 11)

Columns:
['Depot ID', 'Depot Name', 'District', 'Buses Allocated', 'Schedules Allocated', 'Effective KM', 'Passengers', 'Passengers_per_Bus', 'Year', 'Month_Number', 'Predicted EV Suitability Score']

First 5 records:
 Depot ID Depot Name           District  Predicted EV Suitability Score
KSRTC-001      ADOOR     Pathanamthitta                        0.399155
KSRTC-002  ALAPPUZHA          Alappuzha                        0.490111
KSRTC-003      ALUVA          Ernakulam                        0.416494
KSRTC-004   ANKAMALY          Ernakulam                        0.405018
KSRTC-005   ATTINGAL Thiruvananthapuram                        0.441330


In [8]:
# Merge depot ML suitability with terrain information

combined_df = route_df.merge(
    depot_predictions[
        [
            "Depot ID",
            "Predicted EV Suitability Score"
        ]
    ],
    left_on="Depot_ID",
    right_on="Depot ID",
    how="left"
)

print("Combined dataset shape:", combined_df.shape)

print("\nMissing ML predictions:")
print(combined_df["Predicted EV Suitability Score"].isna().sum())

print("\nCombined data:")
print(
    combined_df[
        [
            "Depot_ID",
            "Depot_Name",
            "District",
            "Terrain_Class",
            "Terrain_Score",
            "Terrain_Recommendation",
            "Predicted EV Suitability Score"
        ]
    ].head(70).to_string(index=False)
)

Combined dataset shape: (92, 12)

Missing ML predictions:
0

Combined data:
 Depot_ID      Depot_Name           District Terrain_Class  Terrain_Score Terrain_Recommendation  Predicted EV Suitability Score
KSRTC-001           ADOOR     Pathanamthitta       Rolling           0.75         EV Conditional                        0.399155
KSRTC-002       ALAPPUZHA          Alappuzha          Flat           1.00           EV Favorable                        0.490111
KSRTC-003           ALUVA          Ernakulam  Flat/Rolling           0.90           EV Favorable                        0.416494
KSRTC-004        ANKAMALY          Ernakulam  Flat/Rolling           0.90           EV Favorable                        0.405018
KSRTC-005        ATTINGAL Thiruvananthapuram  Flat/Rolling           0.90           EV Favorable                        0.441330
KSRTC-006 CHADAYAMANGALAM             Kollam  Flat/Rolling           0.90           EV Favorable                        0.495928
KSRTC-007      CHALAK

In [11]:
# Calculate terrain-adjusted EV suitability

combined_df["Terrain_Adjusted_EV_Score"] = (
    combined_df["Predicted EV Suitability Score"]
    * combined_df["Terrain_Score"]
)

# Rank all depots after terrain adjustment
combined_df = combined_df.sort_values(
    "Terrain_Adjusted_EV_Score",
    ascending=False
).reset_index(drop=True)

combined_df["Terrain_Adjusted_Rank"] = (
    combined_df.index + 1
)

print("Top 15 depots after terrain adjustment:\n")

print(
    combined_df[
        [
            "Terrain_Adjusted_Rank",
            "Depot_ID",
            "Depot_Name",
            "District",
            "Terrain_Class",
            "Predicted EV Suitability Score",
            "Terrain_Score",
            "Terrain_Adjusted_EV_Score",
            "Terrain_Recommendation"
        ]
    ].head(90).to_string(index=False)
)

Top 15 depots after terrain adjustment:

 Terrain_Adjusted_Rank  Depot_ID           Depot_Name           District Terrain_Class  Predicted EV Suitability Score  Terrain_Score  Terrain_Adjusted_EV_Score Terrain_Recommendation
                     1 KSRTC-032               KOLLAM             Kollam  Flat/Rolling                        0.559076           0.90                   0.503168           EV Favorable
                     2 KSRTC-024               KANNUR             Kannur  Flat/Rolling                        0.558387           0.90                   0.502549           EV Favorable
                     3 KSRTC-002            ALAPPUZHA          Alappuzha          Flat                        0.490111           1.00                   0.490111           EV Favorable
                     4 KSRTC-026            KASARGODE          Kasaragod  Flat/Rolling                        0.519530           0.90                   0.467577           EV Favorable
                     5 KSRTC-022       

In [12]:
# Check Trivandrum depots after terrain adjustment

trivandrum_check = combined_df[
    combined_df["Depot_Name"].str.contains(
        "TRIVANDRUM",
        case=False,
        na=False
    )
].copy()

print(
    trivandrum_check[
        [
            "Depot_ID",
            "Depot_Name",
            "District",
            "Terrain_Class",
            "Predicted EV Suitability Score",
            "Terrain_Score",
            "Terrain_Adjusted_EV_Score",
            "Terrain_Recommendation",
            "Terrain_Adjusted_Rank"
        ]
    ].sort_values(
        "Terrain_Adjusted_EV_Score",
        ascending=False
    ).to_string(index=False)
)

 Depot_ID           Depot_Name           District Terrain_Class  Predicted EV Suitability Score  Terrain_Score  Terrain_Adjusted_EV_Score Terrain_Recommendation  Terrain_Adjusted_Rank
KSRTC-082 TRIVANDRUM - CENTRAL Thiruvananthapuram  Flat/Rolling                        0.493352            0.9                   0.444017           EV Favorable                     13
KSRTC-083    TRIVANDRUM - CITY Thiruvananthapuram  Flat/Rolling                        0.433003            0.9                   0.389703           EV Favorable                     45


In [15]:
# Final EV transition priority based on observed depot score distribution

q25 = combined_df["Terrain_Adjusted_EV_Score"].quantile(0.25)
q75 = combined_df["Terrain_Adjusted_EV_Score"].quantile(0.75)

def final_ev_decision(score):
    if score >= q75:
        return "EV Preferred"
    elif score >= q25:
        return "EV Conditional"
    else:
        return "Diesel Preferred"

combined_df["Final_EV_Decision"] = (
    combined_df["Terrain_Adjusted_EV_Score"]
    .apply(final_ev_decision)
)

print("Decision thresholds:")
print("EV Preferred      :", round(q75, 4))
print("EV Conditional    :", round(q25, 4))
print("Diesel Preferred  : below", round(q25, 4))

print("\nFinal decision distribution:")
print(
    combined_df["Final_EV_Decision"]
    .value_counts()
)

print("\nFinal decision percentages:")
print(
    combined_df["Final_EV_Decision"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Decision thresholds:
EV Preferred      : 0.4297
EV Conditional    : 0.3207
Diesel Preferred  : below 0.3207

Final decision distribution:
Final_EV_Decision
EV Conditional      46
EV Preferred        23
Diesel Preferred    23
Name: count, dtype: int64

Final decision percentages:
Final_EV_Decision
EV Conditional      50.0
EV Preferred        25.0
Diesel Preferred    25.0
Name: proportion, dtype: float64


In [14]:
# Examine the distribution of terrain-adjusted EV scores

print("Terrain-adjusted score statistics:\n")

print(
    combined_df["Terrain_Adjusted_EV_Score"]
    .describe()
    .round(4)
)

print("\nSelected percentiles:\n")

percentiles = combined_df[
    "Terrain_Adjusted_EV_Score"
].quantile(
    [0.25, 0.50, 0.75]
)

print(percentiles.round(4))

Terrain-adjusted score statistics:

count    92.0000
mean      0.3554
std       0.1070
min       0.0677
25%       0.3207
50%       0.3826
75%       0.4297
max       0.5032
Name: Terrain_Adjusted_EV_Score, dtype: float64

Selected percentiles:

0.25    0.3207
0.50    0.3826
0.75    0.4297
Name: Terrain_Adjusted_EV_Score, dtype: float64


In [16]:
# Display EV Preferred depots

ev_preferred = combined_df[
    combined_df["Final_EV_Decision"] == "EV Preferred"
].copy()

ev_preferred = ev_preferred.sort_values(
    "Terrain_Adjusted_EV_Score",
    ascending=False
)

print("EV Preferred depots:", len(ev_preferred))

print(
    ev_preferred[
        [
            "Terrain_Adjusted_Rank",
            "Depot_ID",
            "Depot_Name",
            "District",
            "Terrain_Class",
            "Predicted EV Suitability Score",
            "Terrain_Score",
            "Terrain_Adjusted_EV_Score",
            "Final_EV_Decision"
        ]
    ].to_string(index=False)
)

EV Preferred depots: 23
 Terrain_Adjusted_Rank  Depot_ID           Depot_Name           District Terrain_Class  Predicted EV Suitability Score  Terrain_Score  Terrain_Adjusted_EV_Score Final_EV_Decision
                     1 KSRTC-032               KOLLAM             Kollam  Flat/Rolling                        0.559076            0.9                   0.503168      EV Preferred
                     2 KSRTC-024               KANNUR             Kannur  Flat/Rolling                        0.558387            0.9                   0.502549      EV Preferred
                     3 KSRTC-002            ALAPPUZHA          Alappuzha          Flat                        0.490111            1.0                   0.490111      EV Preferred
                     4 KSRTC-026            KASARGODE          Kasaragod  Flat/Rolling                        0.519530            0.9                   0.467577      EV Preferred
                     5 KSRTC-022            KANGANGAD          Kasaragod  Flat/Ro

In [17]:
# Validate Diesel Preferred depots

diesel_preferred = combined_df[
    combined_df["Final_EV_Decision"] == "Diesel Preferred"
].copy()

diesel_preferred = diesel_preferred.sort_values(
    "Terrain_Adjusted_EV_Score",
    ascending=True
)

print("Diesel Preferred depots:", len(diesel_preferred))

print(
    diesel_preferred[
        [
            "Terrain_Adjusted_Rank",
            "Depot_ID",
            "Depot_Name",
            "District",
            "Terrain_Class",
            "Predicted EV Suitability Score",
            "Terrain_Score",
            "Terrain_Adjusted_EV_Score",
            "Final_EV_Decision"
        ]
    ].to_string(index=False)
)

Diesel Preferred depots: 23
 Terrain_Adjusted_Rank  Depot_ID      Depot_Name       District Terrain_Class  Predicted EV Suitability Score  Terrain_Score  Terrain_Adjusted_EV_Score Final_EV_Decision
                    92 KSRTC-047     MOOLAMATTOM         Idukki         Steep                        0.338662           0.20                   0.067732  Diesel Preferred
                    91 KSRTC-049          MUNNAR         Idukki         Steep                        0.346322           0.20                   0.069264  Diesel Preferred
                    90 KSRTC-079      THODUPUZHA         Idukki         Steep                        0.346651           0.20                   0.069330  Diesel Preferred
                    89 KSRTC-028      KATTAPPANA         Idukki         Steep                        0.346651           0.20                   0.069330  Diesel Preferred
                    88 KSRTC-051     NEDUMKANDAM         Idukki         Steep                        0.403777           0.

In [18]:
# Inspect low-scoring depots and their operational characteristics

low_score_check = depot_predictions[
    depot_predictions["Depot ID"].isin(
        ["KSRTC-077", "KSRTC-033"]
    )
][
    [
        "Depot ID",
        "Depot Name",
        "District",
        "Buses Allocated",
        "Schedules Allocated",
        "Effective KM",
        "Passengers",
        "Passengers_per_Bus",
        "Predicted EV Suitability Score"
    ]
]

print(low_score_check.to_string(index=False))

 Depot ID Depot Name       District  Buses Allocated  Schedules Allocated  Effective KM  Passengers  Passengers_per_Bus  Predicted EV Suitability Score
KSRTC-033      KONNI Pathanamthitta               31                   27        209907      282322         9107.161290                         0.28135
KSRTC-077 THIRUVALLA Pathanamthitta               31                   27        209727      282081         9099.387097                         0.28135


In [19]:
# Evidence-based terrain profile for depot locations
#
# These classifications represent the dominant operational terrain
# around the depot/corridor, not exact route-level elevation.
#
# Terrain factors are project decision factors:
# Flat = 1.00
# Flat/Rolling = 0.90
# Rolling = 0.75
# Hilly = 0.40
# Steep = 0.20

terrain_profiles = {
    # Predominantly coastal / lowland
    "ALAPPUZHA": ("Flat", 1.00),
    "CHENGANUR": ("Flat", 1.00),
    "CHERTHALA": ("Flat", 1.00),
    "EDATHUVA": ("Flat", 1.00),
    "MAVELIKKARA": ("Flat", 1.00),

    # Predominantly flat / gently undulating
    "ALUVA": ("Flat/Rolling", 0.90),
    "ANKAMALY": ("Flat/Rolling", 0.90),
    "ATTINGAL": ("Flat/Rolling", 0.90),
    "CHADAYAMANGALAM": ("Flat/Rolling", 0.90),
    "CHALAKKUDY": ("Flat/Rolling", 0.90),
    "CHATHANOOR": ("Flat/Rolling", 0.90),
    "CHITTOOR": ("Flat/Rolling", 0.90),
    "ERNAKULAM": ("Flat/Rolling", 0.90),
    "KOLLAM": ("Flat/Rolling", 0.90),
    "KARUNAGAPPALLY": ("Flat/Rolling", 0.90),
    "KOTTARAKKARA": ("Flat/Rolling", 0.90),
    "KULATHUPUZHA": ("Flat/Rolling", 0.90),
    "PALAKKAD": ("Flat/Rolling", 0.90),
    "KOZHIKKODE": ("Flat/Rolling", 0.90),
    "KANNUR": ("Flat/Rolling", 0.90),
    "KASARGODE": ("Flat/Rolling", 0.90),
    "KANGANGAD": ("Flat/Rolling", 0.90),
    "PAYYANNUR": ("Flat/Rolling", 0.90),
    "THALASSERY": ("Flat/Rolling", 0.90),
    "PUNALUR": ("Flat/Rolling", 0.90),
    "PATHANAPURAM": ("Flat/Rolling", 0.90),
    "KANIYAPURAM": ("Flat/Rolling", 0.90),
    "NEDUMANGAD": ("Flat/Rolling", 0.90),
    "KATTAKKADA": ("Flat/Rolling", 0.90),
    "VIZHINJAM": ("Flat/Rolling", 0.90),
    "TRIVANDRUM - CENTRAL": ("Flat/Rolling", 0.90),
    "TRIVANDRUM - CITY": ("Flat/Rolling", 0.90),
    "PIRAVOM": ("Flat/Rolling", 0.90),
    "GURUVAYOOR": ("Flat/Rolling", 0.90),

    # Rolling / foothill / undulating areas
    "ADOOR": ("Rolling", 0.75),
    "CHANGANASSERY": ("Rolling", 0.75),
    "EERATTUPETTAH": ("Rolling", 0.75),
    "ERUMALI": ("Rolling", 0.75),
    "KOTTAYAM": ("Rolling", 0.75),
    "PONKUNNAM": ("Rolling", 0.75),
    "PONNANI": ("Rolling", 0.75),
    "NILAMBUR": ("Rolling", 0.75),
    "PERINTHALMANNA": ("Rolling", 0.75),
    "MALAPPURAM": ("Rolling", 0.75),
    "PANDALAM": ("Rolling", 0.75),
    "RANNI": ("Rolling", 0.75),
    "MALLAPPALLY": ("Rolling", 0.75),
    "PATHANAMTHITTA": ("Rolling", 0.75),
    "PUTHUKKAD": ("Rolling", 0.75),
    "THIRUVALLA": ("Rolling", 0.75),
    "KONNI": ("Rolling", 0.75),

    # Western Ghats / highland terrain
    "KALPETTA": ("Hilly", 0.40),
    "MANANTHAVADY": ("Hilly", 0.40),
    "SULTHAN BATHERY": ("Hilly", 0.40),

    # Steep highland / mountain terrain
    "CHERUTHONI": ("Steep", 0.20),
    "KUMILY": ("Steep", 0.20),
    "KATTAPPANA": ("Steep", 0.20),
    "MUNNAR": ("Steep", 0.20),
    "NEDUMKANDAM": ("Steep", 0.20),
    "THODUPUZHA": ("Steep", 0.20),
    "MOOLAMATTOM": ("Steep", 0.20),
}

In [20]:
# Check terrain coverage

missing_depots = sorted(
    set(combined_df["Depot_Name"].unique())
    - set(terrain_profiles.keys())
)

print("Total depots:", combined_df["Depot_Name"].nunique())
print("Terrain profiles available:", len(terrain_profiles))
print("Missing depot profiles:", len(missing_depots))

if missing_depots:
    print("\nMissing depots:")
    print(missing_depots)
else:
    print("\nAll 92 depots have terrain profiles.")

Total depots: 92
Terrain profiles available: 61
Missing depot profiles: 31

Missing depots:
['HARIPPAD', 'IRINJALAKKUDA', 'KAYAMKULAM', 'KILIMANOOR', 'KODUNGALOOR', 'KOOTHATTUKULAM', 'KOTHAMANGALAM', 'MALA', 'MANNARKKAD', 'MOOVATTUPUZHA', 'NEYYATTINKARA', 'NORTH PARAVUR', 'PALA', 'PALODE', 'PAPPANAMCODE', 'PARASSALA', 'PEROORKKADA', 'PERUMBAVOOR', 'POOVAR', 'THAMARASSERY', 'THIRUVAMBADY', 'THOTTILPALLAM', 'THRISSUR', 'VADAKARA', 'VADAKKANCHERY', 'VAIKKOM', 'VELLANAD', 'VELLARADA', 'VENJARAMOOD', 'VIKAS BHAVAN', 'VITHURA']


In [21]:
# Review the 31 depots not yet covered by the evidence-based terrain profile

missing_terrain_df = combined_df[
    combined_df["Depot_Name"].isin(missing_depots)
][
    [
        "Depot_ID",
        "Depot_Name",
        "District",
        "Terrain_Class",
        "Terrain_Score",
        "Terrain_Recommendation"
    ]
].copy()

print(
    missing_terrain_df
    .sort_values(["District", "Depot_Name"])
    .to_string(index=False)
)

 Depot_ID     Depot_Name           District Terrain_Class  Terrain_Score Terrain_Recommendation
KSRTC-019       HARIPPAD          Alappuzha          Flat           1.00           EV Favorable
KSRTC-029     KAYAMKULAM          Alappuzha          Flat           1.00           EV Favorable
KSRTC-034 KOOTHATTUKULAM          Ernakulam  Flat/Rolling           0.90           EV Favorable
KSRTC-035  KOTHAMANGALAM          Ernakulam  Flat/Rolling           0.90           EV Favorable
KSRTC-048  MOOVATTUPUZHA          Ernakulam  Flat/Rolling           0.90           EV Favorable
KSRTC-054  NORTH PARAVUR          Ernakulam  Flat/Rolling           0.90           EV Favorable
KSRTC-066    PERUMBAVOOR          Ernakulam  Flat/Rolling           0.90           EV Favorable
KSRTC-055           PALA           Kottayam       Rolling           0.75         EV Conditional
KSRTC-086        VAIKKOM           Kottayam       Rolling           0.75         EV Conditional
KSRTC-076   THAMARASSERY          Kozhik

In [22]:
# Create a broader physiographic terrain profile
# based on Kerala's lowland / midland / highland structure.
#
# IMPORTANT:
# This is an additional evidence/context layer.
# It does NOT overwrite the existing Terrain_Class.

physiographic_profiles = {
    "Alappuzha": "Lowland / Coastal",
    "Ernakulam": "Midland / Coastal-to-Midland",
    "Kottayam": "Midland / Foothill",
    "Kozhikode": "Coastal-to-Midland",
    "Palakkad": "Midland / Highland transition",
    "Thiruvananthapuram": "Coastal-to-Midland / Eastern Highland transition",
    "Thrissur": "Coastal-to-Midland",
    "Pathanamthitta": "Rolling Midland / Highland transition",
    "Kollam": "Coastal-to-Midland / Eastern Highland transition",
    "Malappuram": "Coastal-to-Midland / Eastern Highland transition",
    "Kannur": "Coastal-to-Midland / Eastern Highland transition",
    "Kasaragod": "Coastal-to-Midland / Eastern Highland transition",
    "Idukki": "Highland",
    "Wayanad": "Highland"
}

combined_df["Physiographic_Profile"] = (
    combined_df["District"]
    .map(physiographic_profiles)
)

print("Missing physiographic profiles:")
print(
    combined_df["Physiographic_Profile"]
    .isna()
    .sum()
)

print("\nPhysiographic profile distribution:")
print(
    combined_df["Physiographic_Profile"]
    .value_counts()
)

print("\nTerrain + physiographic profile:")
print(
    combined_df[
        [
            "Depot_ID",
            "Depot_Name",
            "District",
            "Terrain_Class",
            "Terrain_Score",
            "Physiographic_Profile"
        ]
    ].head(20).to_string(index=False)
)

Missing physiographic profiles:
0

Physiographic profile distribution:
Physiographic_Profile
Coastal-to-Midland / Eastern Highland transition    36
Coastal-to-Midland                                  12
Highland                                            10
Midland / Coastal-to-Midland                         9
Rolling Midland / Highland transition                8
Lowland / Coastal                                    7
Midland / Foothill                                   6
Midland / Highland transition                        4
Name: count, dtype: int64

Terrain + physiographic profile:
 Depot_ID           Depot_Name           District Terrain_Class  Terrain_Score                            Physiographic_Profile
KSRTC-032               KOLLAM             Kollam  Flat/Rolling            0.9 Coastal-to-Midland / Eastern Highland transition
KSRTC-024               KANNUR             Kannur  Flat/Rolling            0.9 Coastal-to-Midland / Eastern Highland transition
KSRTC-002            AL

In [23]:
# Create a clear terrain interpretation for EV operations

def terrain_nature(terrain):
    if terrain == "Flat":
        return "Low-gradient terrain with minimal terrain-related EV penalty"
    elif terrain == "Flat/Rolling":
        return "Mostly moderate terrain with some rolling sections"
    elif terrain == "Rolling":
        return "Undulating terrain with increased energy demand on gradients"
    elif terrain == "Hilly":
        return "Hilly terrain with higher gradient-related energy demand"
    elif terrain == "Steep":
        return "Steep terrain with high gradient-related energy demand"
    else:
        return "Terrain requires further assessment"

def ev_terrain_implication(terrain):
    if terrain == "Flat":
        return "Highly favorable for EV operation"
    elif terrain == "Flat/Rolling":
        return "Generally favorable; route-level verification recommended"
    elif terrain == "Rolling":
        return "Conditional; evaluate gradients, range and charging"
    elif terrain == "Hilly":
        return "Caution; diesel may be preferable on demanding routes"
    elif terrain == "Steep":
        return "Diesel preferred for demanding operations"
    else:
        return "Further assessment required"

combined_df["Terrain_Nature"] = (
    combined_df["Terrain_Class"].apply(terrain_nature)
)

combined_df["EV_Terrain_Implication"] = (
    combined_df["Terrain_Class"].apply(ev_terrain_implication)
)

print(
    combined_df[
        [
            "Depot_ID",
            "Depot_Name",
            "District",
            "Terrain_Class",
            "Terrain_Score",
            "Terrain_Nature",
            "EV_Terrain_Implication"
        ]
    ].head(20).to_string(index=False)
)

 Depot_ID           Depot_Name           District Terrain_Class  Terrain_Score                                               Terrain_Nature                                    EV_Terrain_Implication
KSRTC-032               KOLLAM             Kollam  Flat/Rolling            0.9           Mostly moderate terrain with some rolling sections Generally favorable; route-level verification recommended
KSRTC-024               KANNUR             Kannur  Flat/Rolling            0.9           Mostly moderate terrain with some rolling sections Generally favorable; route-level verification recommended
KSRTC-002            ALAPPUZHA          Alappuzha          Flat            1.0 Low-gradient terrain with minimal terrain-related EV penalty                         Highly favorable for EV operation
KSRTC-026            KASARGODE          Kasaragod  Flat/Rolling            0.9           Mostly moderate terrain with some rolling sections Generally favorable; route-level verification recommended
KSRTC-022 

In [24]:
# Final terrain-aware EV suitability score

combined_df["Terrain_Adjusted_EV_Score"] = (
    combined_df["Predicted EV Suitability Score"]
    * combined_df["Terrain_Score"]
)

# Rank all 92 depots
combined_df = combined_df.sort_values(
    "Terrain_Adjusted_EV_Score",
    ascending=False
).reset_index(drop=True)

combined_df["Terrain_Adjusted_Rank"] = (
    combined_df.index + 1
)

print("Terrain-aware EV score calculated successfully.")

print("\nScore statistics:")
print(
    combined_df["Terrain_Adjusted_EV_Score"].describe()
)

print("\nTop 15 depots:")
print(
    combined_df[
        [
            "Depot_ID",
            "Depot_Name",
            "District",
            "Terrain_Class",
            "Terrain_Score",
            "Predicted EV Suitability Score",
            "Terrain_Adjusted_EV_Score",
            "Terrain_Adjusted_Rank"
        ]
    ].head(15).to_string(index=False)
)

Terrain-aware EV score calculated successfully.

Score statistics:
count    92.000000
mean      0.355350
std       0.106975
min       0.067732
25%       0.320712
50%       0.382563
75%       0.429672
max       0.503168
Name: Terrain_Adjusted_EV_Score, dtype: float64

Top 15 depots:
 Depot_ID           Depot_Name           District Terrain_Class  Terrain_Score  Predicted EV Suitability Score  Terrain_Adjusted_EV_Score  Terrain_Adjusted_Rank
KSRTC-032               KOLLAM             Kollam  Flat/Rolling            0.9                        0.559076                   0.503168                      1
KSRTC-024               KANNUR             Kannur  Flat/Rolling            0.9                        0.558387                   0.502549                      2
KSRTC-002            ALAPPUZHA          Alappuzha          Flat            1.0                        0.490111                   0.490111                      3
KSRTC-026            KASARGODE          Kasaragod  Flat/Rolling          

In [25]:
# Final EV transition priority classification

q25 = combined_df["Terrain_Adjusted_EV_Score"].quantile(0.25)
q75 = combined_df["Terrain_Adjusted_EV_Score"].quantile(0.75)

def final_ev_priority(score):
    if score >= q75:
        return "EV Preferred"
    elif score >= q25:
        return "EV Conditional"
    else:
        return "Diesel Preferred"

combined_df["Final_EV_Priority"] = (
    combined_df["Terrain_Adjusted_EV_Score"]
    .apply(final_ev_priority)
)

print("Final EV transition priority created.")

print("\nThresholds:")
print(f"Bottom 25% threshold : {q25:.4f}")
print(f"Top 25% threshold    : {q75:.4f}")

print("\nPriority distribution:")
print(
    combined_df["Final_EV_Priority"]
    .value_counts()
)

print("\nFinal top 15:")
print(
    combined_df[
        [
            "Depot_ID",
            "Depot_Name",
            "District",
            "Terrain_Class",
            "Predicted EV Suitability Score",
            "Terrain_Adjusted_EV_Score",
            "Final_EV_Priority",
            "Terrain_Adjusted_Rank"
        ]
    ]
    .head(15)
    .to_string(index=False)
)

Final EV transition priority created.

Thresholds:
Bottom 25% threshold : 0.3207
Top 25% threshold    : 0.4297

Priority distribution:
Final_EV_Priority
EV Conditional      46
EV Preferred        23
Diesel Preferred    23
Name: count, dtype: int64

Final top 15:
 Depot_ID           Depot_Name           District Terrain_Class  Predicted EV Suitability Score  Terrain_Adjusted_EV_Score Final_EV_Priority  Terrain_Adjusted_Rank
KSRTC-032               KOLLAM             Kollam  Flat/Rolling                        0.559076                   0.503168      EV Preferred                      1
KSRTC-024               KANNUR             Kannur  Flat/Rolling                        0.558387                   0.502549      EV Preferred                      2
KSRTC-002            ALAPPUZHA          Alappuzha          Flat                        0.490111                   0.490111      EV Preferred                      3
KSRTC-026            KASARGODE          Kasaragod  Flat/Rolling                  

In [26]:
# Validate terrain-sensitive depots and districts

validation_districts = [
    "Idukki",
    "Wayanad",
    "Pathanamthitta",
    "Thrissur",
    "Thiruvananthapuram"
]

validation_df = combined_df[
    combined_df["District"].isin(validation_districts)
].copy()

validation_df = validation_df.sort_values(
    "Terrain_Adjusted_EV_Score",
    ascending=False
)

print("Terrain-sensitive depot validation")
print("=" * 120)

print(
    validation_df[
        [
            "Depot_ID",
            "Depot_Name",
            "District",
            "Terrain_Class",
            "Terrain_Score",
            "Predicted EV Suitability Score",
            "Terrain_Adjusted_EV_Score",
            "Final_EV_Priority"
        ]
    ].to_string(index=False)
)

print("\nPriority distribution by district:")
print(
    pd.crosstab(
        validation_df["District"],
        validation_df["Final_EV_Priority"]
    )
)

Terrain-sensitive depot validation
 Depot_ID           Depot_Name           District Terrain_Class  Terrain_Score  Predicted EV Suitability Score  Terrain_Adjusted_EV_Score Final_EV_Priority
KSRTC-082 TRIVANDRUM - CENTRAL Thiruvananthapuram  Flat/Rolling           0.90                        0.493352                   0.444017      EV Preferred
KSRTC-023          KANIYAPURAM Thiruvananthapuram  Flat/Rolling           0.90                        0.488812                   0.439930      EV Preferred
KSRTC-050           NEDUMANGAD Thiruvananthapuram  Flat/Rolling           0.90                        0.485984                   0.437386      EV Preferred
KSRTC-027           KATTAKKADA Thiruvananthapuram  Flat/Rolling           0.90                        0.480356                   0.432320      EV Preferred
KSRTC-092            VIZHINJAM Thiruvananthapuram  Flat/Rolling           0.90                        0.480356                   0.432320      EV Preferred
KSRTC-087             VELLANA

In [27]:
# Final practical EV transition decision
#
# The ML + terrain-adjusted score remains the ranking score.
# Terrain class acts as a practical constraint for the final decision.

def practical_ev_decision(row):
    terrain = row["Terrain_Class"]
    score = row["Terrain_Adjusted_EV_Score"]

    # Strong terrain constraint
    if terrain == "Steep":
        return "Diesel Preferred"

    # Hilly terrain is not classified as EV Preferred
    elif terrain == "Hilly":
        return "Diesel Preferred"

    # For less demanding terrain, use the relative score thresholds
    elif score >= q75:
        return "EV Preferred"

    elif score >= q25:
        return "EV Conditional"

    else:
        return "Diesel Preferred"


combined_df["Final_EV_Decision"] = combined_df.apply(
    practical_ev_decision,
    axis=1
)

print("Final practical EV decision created.")

print("\nDecision distribution:")
print(
    combined_df["Final_EV_Decision"].value_counts()
)

print("\nTerrain vs final decision:")
print(
    pd.crosstab(
        combined_df["Terrain_Class"],
        combined_df["Final_EV_Decision"]
    )
)

print("\nWayanad:")
print(
    combined_df[
        combined_df["District"] == "Wayanad"
    ][
        [
            "Depot_ID",
            "Depot_Name",
            "Terrain_Class",
            "Predicted EV Suitability Score",
            "Terrain_Adjusted_EV_Score",
            "Final_EV_Decision"
        ]
    ].to_string(index=False)
)

print("\nIdukki:")
print(
    combined_df[
        combined_df["District"] == "Idukki"
    ][
        [
            "Depot_ID",
            "Depot_Name",
            "Terrain_Class",
            "Predicted EV Suitability Score",
            "Terrain_Adjusted_EV_Score",
            "Final_EV_Decision"
        ]
    ].to_string(index=False)
)

Final practical EV decision created.

Decision distribution:
Final_EV_Decision
EV Conditional      46
EV Preferred        23
Diesel Preferred    23
Name: count, dtype: int64

Terrain vs final decision:
Final_EV_Decision  Diesel Preferred  EV Conditional  EV Preferred
Terrain_Class                                                    
Flat                              0               5             2
Flat/Rolling                      1              35            21
Hilly                             3               0             0
Rolling                          12               6             0
Steep                             7               0             0

Wayanad:
 Depot_ID      Depot_Name Terrain_Class  Predicted EV Suitability Score  Terrain_Adjusted_EV_Score Final_EV_Decision
KSRTC-074 SULTHAN BATHERY         Hilly                        0.533127                   0.213251  Diesel Preferred
KSRTC-021        KALPETTA         Hilly                        0.532980                   0.

In [30]:
# Create final master dataset safely

operational_columns = [
    "Depot ID",
    "Buses Allocated",
    "Schedules Allocated",
    "Effective KM",
    "Passengers",
    "Passengers_per_Bus",
    "Year",
    "Month_Number"
]

# Keep only the required operational information
operational_df = depot_predictions[operational_columns].copy()

# Rename merge key so both datasets use the same column name
operational_df = operational_df.rename(
    columns={"Depot ID": "Depot_ID"}
)

# Merge operational data into the terrain + ML dataset
final_master_df = combined_df.merge(
    operational_df,
    on="Depot_ID",
    how="left"
)

# Organize final columns
final_master_df = final_master_df[
    [
        "Depot_ID",
        "Depot_Name",
        "District",
        "Buses Allocated",
        "Schedules Allocated",
        "Effective KM",
        "Passengers",
        "Passengers_per_Bus",
        "Year",
        "Month_Number",
        "Predicted EV Suitability Score",
        "Physiographic_Profile",
        "Terrain_Class",
        "Terrain_Score",
        "Terrain_Nature",
        "EV_Terrain_Implication",
        "Terrain_Adjusted_EV_Score",
        "Terrain_Adjusted_Rank",
        "Final_EV_Decision"
    ]
].copy()

print("Final master dataset created successfully.")
print("Shape:", final_master_df.shape)

print("\nMissing values:")
print(final_master_df.isnull().sum())

print("\nFirst 10 records:")
print(
    final_master_df.head(10).to_string(index=False)
)

Final master dataset created successfully.
Shape: (92, 19)

Missing values:
Depot_ID                          0
Depot_Name                        0
District                          0
Buses Allocated                   0
Schedules Allocated               0
Effective KM                      0
Passengers                        0
Passengers_per_Bus                0
Year                              0
Month_Number                      0
Predicted EV Suitability Score    0
Physiographic_Profile             0
Terrain_Class                     0
Terrain_Score                     0
Terrain_Nature                    0
EV_Terrain_Implication            0
Terrain_Adjusted_EV_Score         0
Terrain_Adjusted_Rank             0
Final_EV_Decision                 0
dtype: int64

First 10 records:
 Depot_ID     Depot_Name  District  Buses Allocated  Schedules Allocated  Effective KM  Passengers  Passengers_per_Bus  Year  Month_Number  Predicted EV Suitability Score                            Physiograp

In [31]:
# Final integrity and consistency checks

print("=" * 70)
print("TEJAS-EV FINAL DATASET QUALITY CHECK")
print("=" * 70)

# 1. Row count
print("\n1. Row count:")
print("Rows:", len(final_master_df))
print("Expected:", 92)

# 2. Unique depots
print("\n2. Unique depots:")
print("Unique Depot IDs:", final_master_df["Depot_ID"].nunique())
print("Expected:", 92)

# 3. Duplicate Depot IDs
print("\n3. Duplicate Depot IDs:")
print(
    final_master_df["Depot_ID"].duplicated().sum()
)

# 4. Missing values
print("\n4. Total missing values:")
print(
    final_master_df.isnull().sum().sum()
)

# 5. Score validity
print("\n5. Score validity:")

print(
    "ML score outside [0,1]:",
    (
        (final_master_df["Predicted EV Suitability Score"] < 0) |
        (final_master_df["Predicted EV Suitability Score"] > 1)
    ).sum()
)

print(
    "Terrain score outside [0,1]:",
    (
        (final_master_df["Terrain_Score"] < 0) |
        (final_master_df["Terrain_Score"] > 1)
    ).sum()
)

print(
    "Adjusted score outside [0,1]:",
    (
        (final_master_df["Terrain_Adjusted_EV_Score"] < 0) |
        (final_master_df["Terrain_Adjusted_EV_Score"] > 1)
    ).sum()
)

# 6. Rank validation
print("\n6. Rank validation:")

print(
    "Minimum rank:",
    final_master_df["Terrain_Adjusted_Rank"].min()
)

print(
    "Maximum rank:",
    final_master_df["Terrain_Adjusted_Rank"].max()
)

print(
    "Unique ranks:",
    final_master_df["Terrain_Adjusted_Rank"].nunique()
)

# 7. Decision values
print("\n7. Final decision categories:")
print(
    final_master_df["Final_EV_Decision"].value_counts()
)

# 8. Terrain consistency
print("\n8. Terrain consistency:")
print(
    pd.crosstab(
        final_master_df["Terrain_Class"],
        final_master_df["Final_EV_Decision"]
    )
)

# 9. Year check
print("\n9. Scenario year:")
print(
    final_master_df["Year"].value_counts()
)

# 10. Final status
checks_passed = (
    len(final_master_df) == 92
    and final_master_df["Depot_ID"].nunique() == 92
    and final_master_df["Depot_ID"].duplicated().sum() == 0
    and final_master_df.isnull().sum().sum() == 0
    and (
        (final_master_df["Predicted EV Suitability Score"] >= 0) &
        (final_master_df["Predicted EV Suitability Score"] <= 1)
    ).all()
    and (
        (final_master_df["Terrain_Score"] >= 0) &
        (final_master_df["Terrain_Score"] <= 1)
    ).all()
    and (
        (final_master_df["Terrain_Adjusted_EV_Score"] >= 0) &
        (final_master_df["Terrain_Adjusted_EV_Score"] <= 1)
    ).all()
    and final_master_df["Terrain_Adjusted_Rank"].nunique() == 92
)

print("\n" + "=" * 70)

if checks_passed:
    print("ALL FINAL QUALITY CHECKS PASSED")
else:
    print("SOME QUALITY CHECKS REQUIRE ATTENTION")

print("=" * 70)

TEJAS-EV FINAL DATASET QUALITY CHECK

1. Row count:
Rows: 92
Expected: 92

2. Unique depots:
Unique Depot IDs: 92
Expected: 92

3. Duplicate Depot IDs:
0

4. Total missing values:
0

5. Score validity:
ML score outside [0,1]: 0
Terrain score outside [0,1]: 0
Adjusted score outside [0,1]: 0

6. Rank validation:
Minimum rank: 1
Maximum rank: 92
Unique ranks: 92

7. Final decision categories:
Final_EV_Decision
EV Conditional      46
EV Preferred        23
Diesel Preferred    23
Name: count, dtype: int64

8. Terrain consistency:
Final_EV_Decision  Diesel Preferred  EV Conditional  EV Preferred
Terrain_Class                                                    
Flat                              0               5             2
Flat/Rolling                      1              35            21
Hilly                             3               0             0
Rolling                          12               6             0
Steep                             7               0             0

9. Sce

In [32]:
# Export final TEJAS-EV depot analysis

import os

os.makedirs("../outputs", exist_ok=True)

final_output_path = "../outputs/TEJAS_FINAL_DEPOT_ANALYSIS.csv"

final_master_df.to_csv(
    final_output_path,
    index=False
)

print("Final TEJAS-EV dataset exported successfully.")
print("Shape:", final_master_df.shape)
print("File:", final_output_path)

# Verify exported file
check_df = pd.read_csv(final_output_path)

print("\nExport verification:")
print("Rows:", len(check_df))
print("Columns:", len(check_df.columns))
print("Missing values:", check_df.isnull().sum().sum())
print("Unique depots:", check_df["Depot_ID"].nunique())

Final TEJAS-EV dataset exported successfully.
Shape: (92, 19)
File: ../outputs/TEJAS_FINAL_DEPOT_ANALYSIS.csv

Export verification:
Rows: 92
Columns: 19
Missing values: 0
Unique depots: 92
